In [1]:
import numpy as np

In [2]:
urn = ['blue', 'blue', 'blue', 'black', 'black']
print('Sample 1:', np.random.choice(urn, size=2, replace=False))
print('Sample 2:', np.random.choice(urn, size=2, replace=False))

Sample 1: ['black' 'blue']
Sample 2: ['blue' 'black']


In [3]:
n = 10_000
samples = [np.random.choice(urn, size=2, replace=False) for _ in range(n)]
is_matching = [marble1 == marble2 for marble1, marble2 in samples]
print(f"Proportion of samples with matching marbles: {np.mean(is_matching)}")

Proportion of samples with matching marbles: 0.3984


## 3.1.1 표본 추출 설계

In [4]:
from itertools import combinations

In [5]:
all_samples = ["".join(sample) for sample in combinations("ABCDEFG", 3)]
print(all_samples)
print("Number of Samples:", len(all_samples))

['ABC', 'ABD', 'ABE', 'ABF', 'ABG', 'ACD', 'ACE', 'ACF', 'ACG', 'ADE', 'ADF', 'ADG', 'AEF', 'AEG', 'AFG', 'BCD', 'BCE', 'BCF', 'BCG', 'BDE', 'BDF', 'BDG', 'BEF', 'BEG', 'BFG', 'CDE', 'CDF', 'CDG', 'CEF', 'CEG', 'CFG', 'DEF', 'DEG', 'DFG', 'EFG']
Number of Samples: 35


In [6]:
from itertools import permutations
print(["".join(sample) for sample in permutations("ABC")])

['ABC', 'ACB', 'BAC', 'BCA', 'CAB', 'CBA']


## 3.1.2. 표본의 통계 분포

## 3.1.3. 표본 분포 시뮬레이션

In [7]:
urn = [1, 1, 0, 1, 0, 1, 0]

In [8]:
sample = np.random.choice(urn, size=3, replace=False)
print(f"Sample: {sample}")
print(f"Prop Failures: {sample.mean()}")

Sample: [1 1 0]
Prop Failures: 0.6666666666666666


In [9]:
samples = [np.random.choice(urn, size=3, replace=False) for _ in range(10_000)]
prop_failures = [s.mean() for s in samples]

In [10]:
import pandas as pd

In [11]:
unique_els, counts_els = np.unique(prop_failures, return_counts=True)
pd.DataFrame({
    "Proportion of failures": unique_els,
    "Fraction of Samples": counts_els / 10_000,
})

,Proportion of failures,Fraction of Samples
0,0.000000,0.0311
1,0.333333,0.3414
2,0.666667,0.5115
3,1.000000,0.1160


## 3.1.4. 초기하분포 시뮬레이션

In [12]:
simulations_fast = np.random.hypergeometric(
    ngood=4, nbad=3, nsample=3, size=10_000
)
print(simulations_fast)

[2 3 2 ... 2 1 2]


In [13]:
unique_els, counts_els = np.unique(simulations_fast, return_counts=True)
pd.DataFrame({
    "Number of failures": unique_els,
    "Fraction of samples": counts_els / 10_000,
})

,Number of failures,Fraction of samples
0,0,0.0285
1,1,0.3408
2,2,0.5191
3,3,0.1116


## 3.2 예제: 선거 여론조사의 편향과 변동 시뮬레이션

In [14]:
proportions = np.array([0.4818, 0.4746, 1 - (0.4818+0.4746)])
n = 1_500
N = 6_165_478
votes = np.trunc(N * proportions).astype(int)
votes

array([2970527, 2926135,  268814])

In [15]:
from scipy.stats import multivariate_hypergeom

multivariate_hypergeom.rvs(votes, n)

array([707, 736,  57])

In [16]:
multivariate_hypergeom.rvs(votes, n)

array([740, 683,  77])

In [17]:
import numpy as np
from scipy.stats import multivariate_hypergeom

# 데이터 설정
proportions = np.array([0.4818, 0.4746, 1 - (0.4818+0.4746)])
n = 1_500
N = 6_165_478
votes = np.trunc(N * proportions).astype(int)

# [수정된 부분] 
# 반복문(for) 대신 size 파라미터로 한 번에 10만 개 생성
# 결과 shape: (100000, 3)
simulations_matrix = multivariate_hypergeom.rvs(votes, n, size=100_000)

# Numpy 배열 연산으로 한 번에 계산 (Trump - Harris) / n
trump_advantages = (simulations_matrix[:, 0] - simulations_matrix[:, 1]) / n

print(f"평균 우위: {np.mean(trump_advantages):.4f}")
print("계산 완료")

평균 우위: 0.0073
계산 완료


In [18]:
np.mean(trump_advantages > 0)

np.float64(0.60629)

In [19]:
bias = 0.005
proportion_bias = np.array([0.4818 - bias, 0.4747 + bias, 
                            1 - (0.4818+0.4746)])
proportion_bias

array([0.4768, 0.4797, 0.0436])

In [20]:
votes_bias = np.trunc(N * proportion_bias).astype(int)
votes_bias

array([2939699, 2957579,  268814])

In [21]:
def t_adv(votes, n):
    sample_votes = multivariate_hypergeom.rvs(votes, n)
    return (sample_votes[0] - sample_votes[1]) / n

In [22]:
simulations_bias = [t_adv(votes_bias, n) for _ in range(100000)]

In [23]:
np.mean(np.array(simulations_bias) > 0)

np.float64(0.44935)

In [24]:
simulations_big = [t_adv(votes, 12000) for _ in range(100000)]

simulations_bias_big = [t_adv(votes_bias, 12000) for _ in range(100000)]

In [25]:
scenario_no_bias = np.mean(np.array(simulations_big) > 0)
scenario_bias = np.mean(np.array(simulations_bias_big) > 0)

In [26]:
print(scenario_no_bias, scenario_bias)

0.78834 0.37088


## 3.3 예제: 백신 무작위 임상시험 시뮬레이션

In [28]:
simulations_fast = np.random.hypergeometric(ngood=468,
                                            nbad=43270,
                                            nsample=21869,
                                            size=500000)

In [29]:
simulations_fast

array([250, 246, 237, ..., 229, 257, 230], shape=(500000,))

## 3.4. 예시: 대기질 측정

In [30]:
pm = pd.read_csv("pm30.csv")

In [31]:
np.std(pm.diff30)

np.float64(0.6870817156282194)

In [32]:
urn = pm['diff30']

In [33]:
np.random.seed(221212)
sample_err = np.random.choice(urn, size=15, replace=True)
aq_imitate = 11 + sample_err

In [34]:
aq_imitate

array([11.35, 11.  , 10.45, 11.12, 10.81, 11.33, 10.01, 10.87, 11.05,
       11.11, 11.05, 12.53, 11.05, 10.78, 10.81])